# Eksplorasi pasca-G2 (E1-auto, E2, E3) + grafik

**Eksploratori, bukan pengganti G2.** Verdict G2 (FAIL) dan NO-GO tidak berubah. Kalimat wajib bila dilaporkan: *analisis eksploratori pasca-G2, tidak dipre-registrasi sebelum G2* (pre-registrasi: vault `Process/025` bagian 5).

CPU cukup (~1-2 menit). Input data harus ada di Drive: `MyDrive/Thesis/pilot/explore_inputs/` (s4, s5, s6/synth, g2, s7).

In [ ]:
# 1. kode dari GitHub
import os, subprocess
REPO = '/content/thesis-code'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/imamdigmi/tracking-date-palm-loss-gain-pleiades-qatar', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# 2. mount Drive & salin input ke pilot/out/
from google.colab import drive
drive.mount('/content/drive')
import shutil
DRIVE = '/content/drive/MyDrive/Thesis/pilot'
SRC = f'{DRIVE}/explore_inputs'
assert os.path.isdir(SRC), f'input tidak ada: {SRC}'
shutil.copytree(SRC, f'{REPO}/pilot/out', dirs_exist_ok=True)
for d in ['s4/obs.parquet', 's4/sites.parquet', 's5/candidates.parquet', 's6/synth/obs.parquet', 's6/synth/truth_events.csv', 'g2/labels.parquet', 'g2/preds.parquet', 's7/farm_metrics.csv']:
    assert os.path.exists(f'{REPO}/pilot/out/{d}'), d
print('input lengkap')

In [ ]:
# 3. dependensi: Colab sudah punya pandas, numpy, scipy, matplotlib, pyarrow, pyyaml -> pip hanya bila ada yang kurang
import importlib.util
need = {'pandas': 'pandas', 'numpy': 'numpy', 'scipy': 'scipy', 'matplotlib': 'matplotlib', 'pyarrow': 'pyarrow', 'yaml': 'pyyaml'}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run(['pip', 'install', '-q', *missing], check=True)
print('kurang:', missing)

In [ ]:
# 4. jalankan E3, E2, E1-auto (hasil di pilot/out/explore/E{1,2,3}/)
r = subprocess.run(['python', 'pilot/explore_g2.py', 'all'], capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
r.check_returncode()

In [ ]:
# 5. kesimpulan otomatis (aturan pre-registrasi)
import json
for e in ['E3', 'E2', 'E1']:
    print(e, json.dumps(json.load(open(f'pilot/out/explore/{e}/summary.json')), indent=1)[:1500])
    print()

In [ ]:
# 6. grafik (judul & label Bahasa Indonesia)
import sys
sys.path.insert(0, 'pilot')
import explore_plots as pl
from IPython.display import Image, display
FIGS = f'{DRIVE}/explore_outputs/figs'
for p in pl.plot_all('pilot/out', FIGS):
    print(p)
    display(Image(str(p)))

In [ ]:
# 7. salin hasil explore ke Drive
shutil.copytree('pilot/out/explore', f'{DRIVE}/explore_outputs/explore', dirs_exist_ok=True)
print(sorted(os.listdir(f'{DRIVE}/explore_outputs')))